## 1. Connect to your Endpoint


In [12]:
import sys
from pathlib import Path

from IPython.display import Image, display

MODULE = "01-consistent-media-generation-on-serverless-gpus"
candidates = [Path.cwd(), Path.cwd() / "code", Path.cwd() / MODULE / "code"]
found = [p for p in candidates if (p / "common.py").is_file()]
assert found, "Open this notebook from the repository (its code/ folder)"
CODE = found[0].resolve()
sys.path.insert(0, str(CODE))

import importlib

import image
import share_card as share_card_module

# reload(image) keeps share_card already imported by that module.
importlib.reload(share_card_module)
importlib.reload(image)
from common import asset_record, metrics, output_dir, setup_endpoint, wait_ready, write_json
from image import ImageGenerator, cached_image, image_comparison, share_card

BASE_URL, TOKEN = setup_endpoint()

Endpoint configured (***-kB7)


## 2. Wait until it's ready

In [13]:
models = wait_ready(BASE_URL, TOKEN, deadline_s=600)
print("Served models:", [m.get("id") for m in models["data"]])

Served models: ['Efficient-Large-Model/Sana_1600M_1024px_diffusers']


## 3. Generation

In [14]:
STYLE = (
    "professional food photography for a restaurant promotion, warm natural light, "
    "shallow depth of field, clean uncluttered composition, "
    "upper quarter of the image mostly empty space, no people, no text, no logo"
)

scenes = [
    "a cheeseburger with golden fries and a cold drink on a tray",
    "a table set with several shared dishes for two, a salad, pasta and bread",
    "two identical pizzas side by side on a wooden table",
    "a brown paper delivery bag and a food container on a doorstep",
    "a main course plate with a slice of chocolate cake next to it",
    "a large table with many dishes for a family, roast chicken, salads and sides",
    "a bowl of soup, a main course plate and a glass of water on a table",
    "a light lunch bowl, a coffee cup and a notebook on a cafe table",
    "two cocktails and a plate of small appetizers on a bar counter",
    "an open takeaway box with fresh food and a bright napkin",
]

prompts = [f"{scene}. {STYLE}" for scene in scenes]

In [15]:
import statistics

generator = ImageGenerator(BASE_URL, TOKEN)
folder = output_dir(None)

paths, latencies = [], []
for i, prompt in enumerate(prompts, start=1):
    path = generator.generate_image(prompt, out=folder / f"p{i:02d}.png", seed=7)
    paths.append(path)
    latencies.append(round(generator.last_s, 4))
    print(i, latencies[-1], "s")

print("median:", statistics.median(latencies))
print("saved to:", folder)

Generated in 4.8 s
1 4.7587 s
Generated in 1.2 s
2 1.1976 s
Generated in 1.1 s
3 1.1421 s
Generated in 1.1 s
4 1.1481 s
Generated in 1.1 s
5 1.1438 s
Generated in 1.1 s
6 1.118 s
Generated in 1.1 s
7 1.1003 s
Generated in 1.2 s
8 1.1603 s
Generated in 1.1 s
9 1.1323 s
Generated in 1.2 s
10 1.155 s
median: 1.14595
saved to: runs\01-consistent-media-generation-on-serverless-gpus\20261124T103706771477Z


## Cost

In [17]:
import math
import statistics

# --- Inputs (from the Endpoint logs and the Nebius price list) ---
price_per_hour = 0.7412            # USD/h, preemptible L40S (check the price on the day of the run)
price_per_s = price_per_hour / 3600

startup_s = 144.3                # provisioning start -> container started (10:33:47 -> 10:36:11)
load_s = 47.9                    # "ready in 47.9s" in the server log
gap_s = 6.4                      # container started -> server process started
fixed_s = startup_s + gap_s + load_s   # ≈ 198.6 s paid once per run

# --- Results ---
median_s = statistics.median(latencies)   # `latencies` comes from the generation cell
c = median_s * price_per_s                # marginal cost per asset
F = fixed_s * price_per_s                 # fixed cost per run

N = 1000
cost_per_asset = F / N + c

api_price = 0.001                         # fal.ai Sana, $ per asset
breakeven = math.ceil(F / (api_price - c)) if api_price > c else 0

print(f"median generation time: {median_s:.3f} s")
print(f"c (marginal cost per asset): ${c:.6f}")
print(f"F (fixed cost per run): ${F:.4f}")
print(f"projected cost per asset at N={N}: ${cost_per_asset:.6f}")
print(f"breakeven vs API: {breakeven} assets")

median generation time: 1.146 s
c (marginal cost per asset): $0.000236
F (fixed cost per run): $0.0409
projected cost per asset at N=1000: $0.000277
breakeven vs API: 54 assets
